# Can we predict whether a student passes the closing quiz?
Protocol: `protocol.yaml` (group holdout on `student_key`, 5-fold GroupKFold CV on training rows, choose by mean CV ROC AUC, score holdout once).

In [ ]:
import json, hashlib, numpy as np, pandas as pd
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.metrics import roc_auc_score
from sklearn.base import clone

sha = lambda p: hashlib.sha256(open(p, 'rb').read()).hexdigest()
assert sha('data.csv') == '5a0b0221ed641db40322e1a95cae32cc67d3525ecb500813f95d6c109b2a7409'
assert sha('holdout.csv') == 'ff08c7a88fd7e15b17b05a390393550fdf57ea2b6df584f1d0594cc2ec1f00e6'
df = pd.read_csv('data.csv')
hold_rows = pd.read_csv('holdout.csv')['row'].to_numpy()
assert len(df) == 12453 and len(hold_rows) == 2465 and len(set(hold_rows)) == 2465

## Split
Holdout rows come from `holdout.csv` (0-based positions in data.csv). Everything else is training. Students never appear on both sides.

In [ ]:
is_hold = np.zeros(len(df), bool)
is_hold[hold_rows] = True
train_idx = np.where(~is_hold)[0]
hold_idx = np.where(is_hold)[0]
assert not set(df.student_key[train_idx]) & set(df.student_key[hold_idx])
print(len(train_idx), len(hold_idx), df.passed_quiz.iloc[train_idx].mean())

## Features
Excluded: `student_key` (entity key). All remaining columns are known by the end of the session, before the quiz result: `enrol_year`, `school_code`, `prior_score` (placement test, taken earlier), `session_minutes`, `hints_used` (recorded during the session). `school_code` is a numeric code used as-is.
Note: `prior_score` reaches 105, odd for a test score, but not a reason to drop it.

In [ ]:
features = ['enrol_year', 'school_code', 'prior_score', 'session_minutes', 'hints_used']
target = 'passed_quiz'
X = df[features]
y = df[target]
g = df.student_key

## Candidates (named before any scoring)

In [ ]:
candidates = {
    'baseline_prior': DummyClassifier(strategy='prior'),
    'logreg': make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)),
    'random_forest': RandomForestClassifier(n_estimators=300, min_samples_leaf=20, random_state=0, n_jobs=-1),
    'hist_gbm': HistGradientBoostingClassifier(max_depth=3, learning_rate=0.05, max_iter=200, random_state=0),
}
print(list(candidates))

## 5-fold grouped CV on training rows (preprocessing sits inside each pipeline, so it is fit per fold)

In [ ]:
Xtr, ytr, gtr = X.iloc[train_idx], y.iloc[train_idx], g.iloc[train_idx]
folds = list(GroupKFold(n_splits=5).split(Xtr, ytr, gtr))
cv_folds = [[[int(train_idx[i]) for i in tr], [int(train_idx[i]) for i in va]] for tr, va in folds]
cv_scores = {}
for name, m in candidates.items():
    s = []
    for tr, va in folds:
        mm = clone(m).fit(Xtr.iloc[tr], ytr.iloc[tr])
        s.append(float(roc_auc_score(ytr.iloc[va], mm.predict_proba(Xtr.iloc[va])[:, 1])))
    cv_scores[name] = s
cv = pd.DataFrame({k: {'mean': np.mean(v), 'std': np.std(v)} for k, v in cv_scores.items()}).T
cv

## Choose by highest mean CV ROC AUC, refit on all training rows, score holdout once

In [ ]:
chosen = max(cv_scores, key=lambda k: np.mean(cv_scores[k]))
final = clone(candidates[chosen]).fit(Xtr, ytr)
holdout_auc = float(roc_auc_score(y.iloc[hold_idx], final.predict_proba(X.iloc[hold_idx])[:, 1]))
print('chosen:', chosen, '| CV mean AUC: %.4f' % np.mean(cv_scores[chosen]), '| holdout AUC: %.4f' % holdout_auc)

In [ ]:
evidence = {'features': features, 'candidates': list(candidates), 'cv_folds': cv_folds, 'cv_scores': cv_scores,
            'chosen': chosen, 'holdout_rows': [int(i) for i in hold_idx], 'holdout_scores': {chosen: {'roc_auc': holdout_auc}}}
json.dump(evidence, open('evidence.json', 'w'))

## Conclusion
See the CV table and holdout ROC AUC above. The holdout AUC is the honest estimate; a value near 0.5 would mean the outcome is not predictable from these features.